# AC One — adjustment-agent smoke (indexed scale, 3 rows)

Exercises the Case B toolbelt seam on three CSV rows without touching the
36-row backtest or its prediction cache. Two configurations run on the same
rows: `tools=()` (no evidence, must return zero adjustment) and
`tools=[news_search()]` (cutoff-verified proxy search).

Offline proof (always runs): the prompt JSON contains no `actual_*` key, the
baseline in the payload is the CSV `forecast_indexed`, and the Python cap
clips an over-cap answer. Live proof (needs `OPENAI_API_KEY` /
`OPENAI_BASE_URL` for the Vector proxy): six agent calls, each checked for
baseline echo and `|adjustment_pct| <= cap`.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve()
while not (ROOT / "implementations" / "ac_one").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env", override=False)

from ac_one.analyst_agent import (
    FuelForecastPromptBuilder,
    build_fuel_adjustment_config,
    build_fuel_adjustment_predictor,
    clip_adjustments,
    news_search,
)
from ac_one.data import build_ac_one_service, load_forecast_data
from ac_one.predictors import deterministic_payload
from ac_one.specs import build_backtest_specs, load_experiment_spec


SCALE = "indexed"
CAP_PCT = 20.0
RUN_LIVE = bool(os.getenv("OPENAI_API_KEY"))  # six proxy calls; flip to False to force offline-only
AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # advanced alternative

# (station, target month, horizon) — origin is target minus horizon months, as in data.py.
SMOKE_ROWS = [("STATION_A", "2026-01-01", 1), ("STATION_B", "2026-03-01", 1), ("STATION_A", "2026-06-01", 1)]

data = load_forecast_data()
service = build_ac_one_service(data)
specs = build_backtest_specs(load_experiment_spec(), data, forecast_scale=SCALE)
print("RUN_LIVE =", RUN_LIVE, "| scale =", SCALE, "| cap_pct =", CAP_PCT)

RUN_LIVE = False | scale = indexed | cap_pct = 20.0


## 1. Offline proof: prompt is outcome-free and anchored on the CSV forecast

In [2]:
def smoke_cases():
    for station, target, horizon in SMOKE_ROWS:
        task = specs[f"{station.lower()}_{horizon}m"].task
        origin = (pd.Timestamp(target) - pd.DateOffset(months=horizon)).to_pydatetime()
        row = data[
            (data.station == station) & (data.horizon_date == pd.Timestamp(target)) & (data.month_horizon == horizon)
        ].iloc[0]
        yield station, target, horizon, task, origin, float(row[f"forecast_{SCALE}"])


def all_keys(node):
    if isinstance(node, dict):
        return set(node) | {k for v in node.values() for k in all_keys(v)}
    if isinstance(node, list):
        return {k for v in node for k in all_keys(v)}
    return set()


builder = FuelForecastPromptBuilder(data, forecast_scale=SCALE, cap_pct=CAP_PCT)
for station, target, horizon, task, origin, csv_forecast in smoke_cases():
    payload = json.loads(builder(task=task, context=service.context(origin)))
    leaked = {k for k in all_keys(payload) if k.startswith("actual_") and k != "actual_is_unavailable"}
    assert not leaked, leaked
    assert payload["external_model"]["forecast"] == csv_forecast
    assert payload["constraints"]["max_absolute_adjustment_pct"] == CAP_PCT
    print(
        f"{station} target={target} h={horizon} as_of={payload['as_of']} "
        f"baseline={payload['external_model']['forecast']} actual_keys={sorted(leaked)} OK"
    )

STATION_A target=2026-01-01 h=1 as_of=2025-12-01 baseline=100.0 actual_keys=[] OK
STATION_B target=2026-03-01 h=1 as_of=2026-02-01 baseline=106.7 actual_keys=[] OK
STATION_A target=2026-06-01 h=1 as_of=2026-05-01 baseline=94.05 actual_keys=[] OK


## 2. Offline proof: the cap is enforced in Python, not only by the prompt

In [3]:
from datetime import datetime

from aieng.forecasting.evaluation import Prediction


over_cap = Prediction(
    predictor_id="smoke",
    task_id="smoke",
    issued_at=datetime(2026, 1, 1),
    as_of=datetime(2026, 1, 1),
    forecast_date=datetime(2026, 2, 1),
    payload=deterministic_payload(135.0),
    metadata={"baseline_forecast": 100.0, "adjustment_pct": 35.0},
)
clip_adjustments([over_cap], cap_pct=CAP_PCT)
print(
    {k: over_cap.metadata[k] for k in ("raw_adjustment_pct", "adjustment_pct", "cap_applied", "cap_pct")},
    "point =",
    over_cap.payload.point_forecast,
)
assert abs(over_cap.metadata["adjustment_pct"]) <= CAP_PCT and over_cap.payload.point_forecast == 120.0

{'raw_adjustment_pct': 35.00000000000001, 'adjustment_pct': 20.0, 'cap_applied': True, 'cap_pct': 20.0} point = 120.0


## 3. Live smoke: `tools=()` then `tools=[news_search()]`

Each configuration is built through the same seam Trang's notebook uses.
Predictions are not cached; nothing here feeds the 36-row comparison.

In [4]:
results = []
if not RUN_LIVE:
    print("Skipped: no OPENAI_API_KEY in the environment. The code path above is the offline proof.")
else:
    for label, tools in (("no_tools", ()), ("news_search", [news_search()])):
        config = build_fuel_adjustment_config(model=AGENT_MODEL, forecast_scale=SCALE, tools=tools)
        agent = build_fuel_adjustment_predictor(data, forecast_scale=SCALE, config=config, cap_pct=CAP_PCT)
        for station, target, horizon, task, origin, csv_forecast in smoke_cases():
            preds = agent.predict(task, service.context(origin))
            assert len(preds) == 1, preds
            m = preds[0].metadata
            assert m["baseline_forecast"] == csv_forecast
            assert abs(m["adjustment_pct"]) <= CAP_PCT
            results.append(
                {
                    "config": label,
                    "tools_enabled": m["tools_enabled"],
                    "station": station,
                    "target": target,
                    "h": horizon,
                    "baseline": m["baseline_forecast"],
                    "point": preds[0].payload.point_forecast,
                    "adjustment_pct": round(m["adjustment_pct"], 3),
                    "cap_applied": m["cap_applied"],
                    "n_sources": len(m["source_urls"]),
                    "trace": m.get("langfuse_trace_url", ""),
                }
            )
    print(pd.DataFrame(results).to_string())

Skipped: no OPENAI_API_KEY in the environment. The code path above is the offline proof.
